# 🧹 Brazilian E-Commerce (Olist) - Data Cleaning & Preprocessing Pipeline

## 🎯 Mục tiêu
Dựa trên các phát hiện và phân tích chuyên sâu tại [undertand_data.ipynb](undertand_data.ipynb), notebook này triển khai toàn diện quy trình **làm sạch và chuẩn hóa dữ liệu (Data Cleaning & Preprocessing)** cho 9 bảng Olist E-commerce.

### 📌 Các vấn đề trọng tâm được xử lý theo lộ trình:
1. **Ép kiểu dữ liệu thời gian (`datetime64[ns]`):** Chuẩn hóa tất cả các cột ngày/thời gian trên `orders`, `order_items`, và `reviews`.
2. **Xử lý giá trị khuyết (`null`):**
   - Bảng `reviews`: Điền `"No Title"` và `"No Message"` cho các nhận xét không có văn bản.
   - Bảng `products`: Điền `"unknown"` cho 610 sản phẩm khuyết danh mục, điền giá trị trung vị (`median`) cho trọng lượng/kích thước và điền `0` cho số lượng ảnh/độ dài văn bản (ép kiểu về `int64`).
3. **Xử lý bất thường thanh toán (`payments`):**
   - Loại bỏ 3 dòng thanh toán có loại `payment_type = 'not_defined'` (đều có giá trị thanh toán 0 đồng của các đơn bị hủy).
   - Chuẩn hóa số kỳ trả góp `payment_installments` (thay thế giá trị 0 thành 1 kỳ).
4. **Đồng bộ bảng dịch thuật (`category_translation`):**
   - Bổ sung 2 danh mục bị thiếu trong bảng dịch nhưng có sản phẩm thực tế: `'pc_gamer'` và `'portateis_cozinha_e_preparadores_de_alimentos'`.
   - Bổ sung danh mục `'unknown'` để đảm bảo toàn vẹn khóa ngoại (Foreign Key) khi join với `products`.
5. **Khử trùng lặp & lọc dị biệt địa lý (`geolocation`):**
   - Khử bỏ **261,831 dòng trùng lặp 100% (26.18%)**.
   - Lọc bỏ 33 tọa độ ngoại lai nằm ngoài lãnh thổ Brazil (Vĩ độ ngoài [-34, +5.5], Kinh độ ngoài [-74, -34]).
6. **Chuẩn hóa chuỗi văn bản:** Đưa tên thành phố về chữ thường (lowercase, trimmed) và mã bang về chữ in hoa (uppercase).
7. **Kiểm tra toàn vẹn tham chiếu (Referential Integrity Check):** Đảm bảo tất cả các khóa ngoại (FK) đều có khóa chính (PK) hợp lệ tương ứng trước khi nạp vào PostgreSQL.
8. **Xuất bản dữ liệu sạch (`data/clean/*.csv`):** Lưu trữ định dạng CSV chuẩn phục vụ `database/load_to_postgres.py`.


In [1]:
import os
import sys
import pandas as pd
import numpy as np
import warnings
from IPython.display import display, Markdown

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

print("Libraries and configurations successfully initialized!")

Libraries and configurations successfully initialized!


## 1. Nạp Dữ Liệu Thô (Load Raw Datasets)

Nạp 9 tập tin CSV từ thư mục `../data/raw/` vào các DataFrame riêng biệt.


In [2]:
raw_dir = "../data/raw"

customers_df = pd.read_csv(os.path.join(raw_dir, "olist_customers_dataset.csv"))
orders_df = pd.read_csv(os.path.join(raw_dir, "olist_orders_dataset.csv"))
order_items_df = pd.read_csv(os.path.join(raw_dir, "olist_order_items_dataset.csv"))
payments_df = pd.read_csv(os.path.join(raw_dir, "olist_order_payments_dataset.csv"))
reviews_df = pd.read_csv(os.path.join(raw_dir, "olist_order_reviews_dataset.csv"))
products_df = pd.read_csv(os.path.join(raw_dir, "olist_products_dataset.csv"))
sellers_df = pd.read_csv(os.path.join(raw_dir, "olist_sellers_dataset.csv"))
geolocation_df = pd.read_csv(os.path.join(raw_dir, "olist_geolocation_dataset.csv"))
category_df = pd.read_csv(os.path.join(raw_dir, "product_category_name_translation.csv"))

print("All 9 raw datasets loaded successfully:")
for name, df in [
    ("customers", customers_df), ("orders", orders_df), ("order_items", order_items_df),
    ("payments", payments_df), ("reviews", reviews_df), ("products", products_df),
    ("sellers", sellers_df), ("geolocation", geolocation_df), ("category_translation", category_df)
]:
    print(f" - {name:<22}: {df.shape[0]:>10,d} rows | {df.shape[1]:>2d} columns")

All 9 raw datasets loaded successfully:
 - customers             :     99,441 rows |  5 columns
 - orders                :     99,441 rows |  8 columns
 - order_items           :    112,650 rows |  7 columns
 - payments              :    103,886 rows |  5 columns
 - reviews               :     99,224 rows |  7 columns
 - products              :     32,951 rows |  9 columns
 - sellers               :      3,095 rows |  4 columns
 - geolocation           :  1,000,163 rows |  5 columns
 - category_translation  :         71 rows |  2 columns


## 2. Làm Sạch Bảng `orders`

### 📋 Các thao tác thực hiện:
- Chuyển đổi 5 cột thời gian từ chuỗi ký tự (`object`) sang định dạng chuẩn `datetime64[ns]`.
- Giữ nguyên các giá trị `NaT` ở `order_delivered_customer_date` cho các đơn hàng chưa giao hoặc bị hủy (được chứng minh là thiếu sót hợp lý theo nghiệp vụ tại `undertand_data.ipynb`).
- Xác thực tính hợp lệ của thời gian mua hàng vs thời gian dự kiến giao hàng (`purchase <= estimated`).


In [3]:
date_columns_orders = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

# Chuyển đổi sang datetime64[ns]
for col in date_columns_orders:
    orders_df[col] = pd.to_datetime(orders_df[col], errors='coerce')

# Xác thực kiểu dữ liệu và số lượng non-null
display(pd.DataFrame({
    "Column": date_columns_orders,
    "Data Type": orders_df[date_columns_orders].dtypes.astype(str),
    "Non-Null Count": orders_df[date_columns_orders].notnull().sum().values,
    "Null Count (NaT)": orders_df[date_columns_orders].isnull().sum().values
}))

print(f"[OK] Orders dataset cleaned. Total records: {len(orders_df):,d}")

,Column,Data Type,Non-Null Count,Null Count (NaT)
order_purchase_timestamp,order_purchase_timestamp,datetime64[ns],99441,0
order_approved_at,order_approved_at,datetime64[ns],99281,160
order_delivered_carrier_date,order_delivered_carrier_date,datetime64[ns],97658,1783
order_delivered_customer_date,order_delivered_customer_date,datetime64[ns],96476,2965
order_estimated_delivery_date,order_estimated_delivery_date,datetime64[ns],99441,0


[OK] Orders dataset cleaned. Total records: 99,441


## 3. Làm Sạch Bảng `order_items`

### 📋 Các thao tác thực hiện:
- Ép kiểu cột thời hạn vận chuyển `shipping_limit_date` sang `datetime64[ns]` (ở bản gốc bị bỏ sót dạng `object`).
- Kiểm tra miền giá trị giá bán (`price > 0`) và cước vận chuyển (`freight_value >= 0`).
- Xác nhận cặp khóa chính phức hợp `(order_id, order_item_id)` duy nhất 100%.


In [4]:
# Ép kiểu thời gian hạn giao hàng
order_items_df['shipping_limit_date'] = pd.to_datetime(order_items_df['shipping_limit_date'], errors='coerce')

# Kiểm tra tính hợp lệ của số liệu tiền tệ
invalid_price = (order_items_df['price'] <= 0).sum()
invalid_freight = (order_items_df['freight_value'] < 0).sum()
free_shipping_count = (order_items_df['freight_value'] == 0).sum()

print(f"Price <= 0 count: {invalid_price}")
print(f"Freight < 0 count: {invalid_freight}")
print(f"Free shipping count (freight == 0): {free_shipping_count:,d}")

display(order_items_df[['price', 'freight_value']].describe().T)
print(f"[OK] Order Items dataset cleaned. Total records: {len(order_items_df):,d}")

Price <= 0 count: 0
Freight < 0 count: 0
Free shipping count (freight == 0): 383


,count,mean,std,min,25%,50%,75%,max
price,112650.00,120.65,183.63,0.85,39.90,74.99,134.90,6735.00
freight_value,112650.00,19.99,15.81,0.00,13.08,16.26,21.15,409.68


[OK] Order Items dataset cleaned. Total records: 112,650


## 4. Làm Sạch Bảng `payments`

### 📋 Các thao tác thực hiện:
- Ở file `undertand_data.ipynb`, ta phát hiện:
  - Có 3 dòng có `payment_type = 'not_defined'` với số tiền `payment_value = 0.0` (thuộc các đơn hàng đã bị hủy `canceled`). Ta sẽ lọc bỏ 3 dòng này.
  - Có 2 giao dịch có số kỳ trả góp `payment_installments = 0`. Ta chuẩn hóa tối thiểu là 1 kỳ.
- Xác nhận cặp khóa chính `(order_id, payment_sequential)`.


In [5]:
initial_count = len(payments_df)

# Lọc bỏ các dòng thanh toán không xác định có giá trị 0 đồng
payments_clean_df = payments_df[payments_df['payment_type'] != 'not_defined'].copy()

# Chuẩn hóa số kỳ trả góp tối thiểu là 1
payments_clean_df.loc[payments_clean_df['payment_installments'] == 0, 'payment_installments'] = 1

dropped_rows = initial_count - len(payments_clean_df)
print(f"Dropped {dropped_rows} invalid rows with payment_type='not_defined'")

display(Markdown("#### Phân bố phương thức thanh toán sau làm sạch:"))
display(payments_clean_df['payment_type'].value_counts().to_frame("Count"))

print(f"[OK] Payments dataset cleaned. Remaining records: {len(payments_clean_df):,d}")

Dropped 3 invalid rows with payment_type='not_defined'


#### Phân bố phương thức thanh toán sau làm sạch:

,Count
payment_type,
credit_card,76795
boleto,19784
voucher,5775
debit_card,1529


[OK] Payments dataset cleaned. Remaining records: 103,883


## 5. Làm Sạch Bảng `reviews`

### 📋 Các thao tác thực hiện:
- Điền giá trị mặc định `"No Title"` cho `review_comment_title` (88.34% null).
- Điền giá trị mặc định `"No Message"` cho `review_comment_message` (58.70% null).
- Ép kiểu 2 cột thời gian `review_creation_date` và `review_answer_timestamp` sang `datetime64[ns]`.
- Giữ nguyên cặp khóa phức hợp `(review_id, order_id)` để đảm bảo độ duy nhất 100%.


In [6]:
# Điền khuyết thiếu cho tiêu đề và nội dung nhận xét
reviews_df['review_comment_title'] = reviews_df['review_comment_title'].fillna('No Title')
reviews_df['review_comment_message'] = reviews_df['review_comment_message'].fillna('No Message')

# Ép kiểu thời gian
reviews_df['review_creation_date'] = pd.to_datetime(reviews_df['review_creation_date'], errors='coerce')
reviews_df['review_answer_timestamp'] = pd.to_datetime(reviews_df['review_answer_timestamp'], errors='coerce')

# Xác nhận còn cột nào bị missing không
print(f"Missing values remaining in reviews: {reviews_df.isnull().sum().sum()}")
display(reviews_df.head(3))
print(f"[OK] Reviews dataset cleaned. Total records: {len(reviews_df):,d}")

Missing values remaining in reviews: 0


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,No Title,No Message,2018-01-18,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,No Title,No Message,2018-03-10,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,No Title,No Message,2018-02-17,2018-02-18 14:36:24


[OK] Reviews dataset cleaned. Total records: 99,224


## 6. Làm Sạch Bảng `products` & `category_translation`

### 📋 Các thao tác thực hiện:
- **Đồng bộ danh mục dịch thuật:** Bổ sung 3 bản ghi vào `category_translation`:
  1. `'pc_gamer'` -> `'pc_gamer'`
  2. `'portateis_cozinha_e_preparadores_de_alimentos'` -> `'small_appliances_kitchen_and_food_preparers'`
  3. `'unknown'` -> `'unknown'` (cho các sản phẩm thiếu danh mục)
- **Bảng `products`:**
  - Điền danh mục khuyết bằng `'unknown'`.
  - Điền `0` cho `product_name_lenght`, `product_description_lenght`, `product_photos_qty` và ép kiểu về `int64` (phù hợp với kiểu số nguyên trong PostgreSQL).
  - Điền giá trị trung vị (`median`) cho các thuộc tính kích thước/trọng lượng (`product_weight_g`, `product_length_cm`, `product_height_cm`, `product_width_cm`).


In [7]:
# 1. Bổ sung các danh mục thiếu vào category_translation
additional_translations = pd.DataFrame([
    {
        'product_category_name': 'pc_gamer',
        'product_category_name_english': 'pc_gamer'
    },
    {
        'product_category_name': 'portateis_cozinha_e_preparadores_de_alimentos',
        'product_category_name_english': 'small_appliances_kitchen_and_food_preparers'
    },
    {
        'product_category_name': 'unknown',
        'product_category_name_english': 'unknown'
    }
])

category_clean_df = pd.concat([category_df, additional_translations], ignore_index=True)
category_clean_df.drop_duplicates(subset=['product_category_name'], inplace=True)

# 2. Làm sạch products
products_df['product_category_name'] = products_df['product_category_name'].fillna('unknown')

int_cols = ['product_name_lenght', 'product_description_lenght', 'product_photos_qty']
for col in int_cols:
    products_df[col] = products_df[col].fillna(0).astype('int64')

physical_cols = ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
for col in physical_cols:
    products_df[col] = products_df[col].fillna(products_df[col].median())

# Kiểm tra độ tương thích 100% giữa products và category_translation
missing_translations = set(products_df['product_category_name']) - set(category_clean_df['product_category_name'])
print(f"Product categories missing in translation table: {len(missing_translations)}")

print(f"[OK] Category translation cleaned: {len(category_clean_df)} categories.")
print(f"[OK] Products cleaned: {len(products_df)} products (0 missing values).")

Product categories missing in translation table: 0
[OK] Category translation cleaned: 74 categories.
[OK] Products cleaned: 32951 products (0 missing values).


## 7. Làm Sạch Bảng `sellers` & `customers`

### 📋 Các thao tác thực hiện:
- Chuẩn hóa chuỗi văn bản: Cắt bỏ khoảng trắng thừa (`strip()`), đưa tên thành phố về chữ thường (`lower()`) và mã bang về chữ in hoa (`upper()`).
- Kiểm tra tính duy nhất của khóa chính: `customer_id` và `seller_id`.


In [8]:
# Chuẩn hóa văn bản customers
customers_df['customer_city'] = customers_df['customer_city'].str.strip().str.lower()
customers_df['customer_state'] = customers_df['customer_state'].str.strip().str.upper()

# Chuẩn hóa văn bản sellers
sellers_df['seller_city'] = sellers_df['seller_city'].str.strip().str.lower()
sellers_df['seller_state'] = sellers_df['seller_state'].str.strip().str.upper()

print(f"Customer PK duplicates: {customers_df.duplicated(subset=['customer_id']).sum()}")
print(f"Seller PK duplicates: {sellers_df.duplicated(subset=['seller_id']).sum()}")
print(f"[OK] Customers: {len(customers_df):,d} records | Sellers: {len(sellers_df):,d} records.")

Customer PK duplicates: 0
Seller PK duplicates: 0
[OK] Customers: 99,441 records | Sellers: 3,095 records.


## 8. Làm Sạch Bảng `geolocation`

### 📋 Các thao tác thực hiện:
- Ở file `undertand_data.ipynb`, bảng này có tới **261,831 dòng trùng lặp 100% (26.18%)** và tồn tại các tọa độ ngoài lãnh thổ Brazil.
- Loại bỏ toàn bộ các dòng trùng lặp bằng `drop_duplicates()`.
- Lọc bỏ các tọa độ ngoại lai ngoài Brazil: Vĩ độ nằm trong khoảng [-34.0, +5.5] và Kinh độ nằm trong khoảng [-74.0, -34.0].
- Chuẩn hóa tên thành phố (`lower`, `strip`) và mã bang (`upper`, `strip`).


In [9]:
initial_geo_rows = len(geolocation_df)

# Khử trùng lặp
geo_clean_df = geolocation_df.drop_duplicates().copy()
dups_removed = initial_geo_rows - len(geo_clean_df)

# Lọc tọa độ hợp lệ trên lãnh thổ Brazil
valid_lat = (geo_clean_df['geolocation_lat'] >= -34.0) & (geo_clean_df['geolocation_lat'] <= 5.5)
valid_lng = (geo_clean_df['geolocation_lng'] >= -74.0) & (geo_clean_df['geolocation_lng'] <= -34.0)
geo_clean_df = geo_clean_df[valid_lat & valid_lng]

# Chuẩn hóa văn bản
geo_clean_df['geolocation_city'] = geo_clean_df['geolocation_city'].str.strip().str.lower()
geo_clean_df['geolocation_state'] = geo_clean_df['geolocation_state'].str.strip().str.upper()

outliers_removed = initial_geo_rows - dups_removed - len(geo_clean_df)
print(f"Duplicate rows removed: {dups_removed:,d}")
print(f"Geographical outlier coordinates removed: {outliers_removed:,d}")
print(f"[OK] Geolocation cleaned. Reduced from {initial_geo_rows:,d} to {len(geo_clean_df):,d} rows.")

Duplicate rows removed: 261,831
Geographical outlier coordinates removed: 33
[OK] Geolocation cleaned. Reduced from 1,000,163 to 738,299 rows.


## 9. Kiểm Tra Toàn Vẹn Tham Chiếu (Referential Integrity Check)

Kiểm tra đối chiếu các ràng buộc khóa ngoại (Foreign Keys) giữa các bảng sạch trước khi lưu trữ:
- Mọi đơn hàng trong `order_items` đều tồn tại trong `orders`.
- Mọi sản phẩm trong `order_items` đều tồn tại trong `products`.
- Mọi người bán trong `order_items` đều tồn tại trong `sellers`.
- Mọi đơn hàng trong `payments` đều tồn tại trong `orders`.
- Mọi đơn hàng trong `reviews` đều tồn tại trong `orders`.
- Mọi danh mục trong `products` đều tồn tại trong `category_translation`.


In [10]:
checks = [
    ("order_items -> orders", set(order_items_df['order_id']) - set(orders_df['order_id'])),
    ("order_items -> products", set(order_items_df['product_id']) - set(products_df['product_id'])),
    ("order_items -> sellers", set(order_items_df['seller_id']) - set(sellers_df['seller_id'])),
    ("payments -> orders", set(payments_clean_df['order_id']) - set(orders_df['order_id'])),
    ("reviews -> orders", set(reviews_df['order_id']) - set(orders_df['order_id'])),
    ("products -> category_translation", set(products_df['product_category_name']) - set(category_clean_df['product_category_name']))
]

integrity_results = []
for relation, violations in checks:
    integrity_results.append({
        "Foreign Key Relationship": relation,
        "Violations (Orphan Records)": len(violations),
        "Status": "Passed" if len(violations) == 0 else "Failed"
    })

integrity_df = pd.DataFrame(integrity_results)
display(integrity_df)

,Foreign Key Relationship,Violations (Orphan Records),Status
0,order_items -> orders,0,Passed
1,order_items -> products,0,Passed
2,order_items -> sellers,0,Passed
3,payments -> orders,0,Passed
4,reviews -> orders,0,Passed
5,products -> category_translation,0,Passed


## 10. Xuất Dữ Liệu Sạch (Export Clean Datasets)

Lưu toàn bộ 9 tập tin sạch vào thư mục `../data/clean/` đúng chuẩn tên file được sử dụng bởi `database/load_to_postgres.py`.


In [11]:
clean_dir = "../data/clean"
os.makedirs(clean_dir, exist_ok=True)

export_files = {
    "customers_clean.csv": customers_df,
    "orders_clean.csv": orders_df,
    "order_items_clean.csv": order_items_df,
    "payments_clean.csv": payments_clean_df,
    "reviews_clean.csv": reviews_df,
    "products_clean.csv": products_df,
    "sellers_clean.csv": sellers_df,
    "geolocation_clean.csv": geo_clean_df,
    "category_translation_clean.csv": category_clean_df
}

export_summary = []
for filename, df in export_files.items():
    dest_path = os.path.join(clean_dir, filename)
    df.to_csv(dest_path, index=False)
    file_size_mb = os.path.getsize(dest_path) / (1024 * 1024)
    export_summary.append({
        "Cleaned File": filename,
        "Rows": f"{len(df):,d}",
        "Columns": df.shape[1],
        "Size (MB)": round(file_size_mb, 2),
        "Nulls Remaining": df.isnull().sum().sum(),
        "Export Status": "Saved"
    })

export_df = pd.DataFrame(export_summary)
display(export_df)
print("\nAll 9 cleaned datasets exported successfully to ../data/clean/!")

,Cleaned File,Rows,Columns,Size (MB),Nulls Remaining,Export Status
0,customers_clean.csv,"99,441",5,8.26,0,Saved
1,orders_clean.csv,"99,441",8,15.84,4908,Saved
2,order_items_clean.csv,"112,650",7,14.33,0,Saved
3,payments_clean.csv,"103,883",5,5.47,0,Saved
4,reviews_clean.csv,"99,224",7,14.63,0,Saved
5,products_clean.csv,"32,951",9,2.52,0,Saved
6,sellers_clean.csv,"3,095",4,0.16,0,Saved
7,geolocation_clean.csv,"738,299",5,42.03,0,Saved
8,category_translation_clean.csv,74,2,0.00,0,Saved



All 9 cleaned datasets exported successfully to ../data/clean/!
